In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt


# stats packages
import scipy.stats as stats
import statsmodels.api as sm
from statsmodels.formula.api import ols, mixedlm
from statsmodels.stats.anova import AnovaRM
import cv2
import os


# set save loaction for figures
fig_save_path = "/Users/thomassainsbury/Documents/Mathis_lab/Aug_Reg/AR_plots_new/"
data_path = "/Volumes/Elements/AR_sync_videos/vegavis_shapes/"
mouse_name = "Vegavis"
date = "2024-09-06"

In [2]:
cd ..

/Users/thomassainsbury/Documents/Mathis_lab/Mathis_lab_code/FreelyMovingVR4Mice/dj_pipeline


In [3]:
%run env_tom.py
%run run.py connect

2024-09-23 18:51:34,131::INFO::settings.py::Setting loglevel to INFO
2024-09-23 18:51:34,132::INFO::settings.py::Setting stores to {}
2024-09-23 18:51:34,132::INFO::settings.py::Setting database.misc.schema_prefix to 
2024-09-23 18:51:34,132::INFO::settings.py::Setting database.misc.create_tables to True
2024-09-23 18:51:34,133::INFO::settings.py::Setting enable_python_native_blobs to True
2024-09-23 18:51:34,133::INFO::settings.py::Setting database.host to 128.178.51.167:3309
2024-09-23 18:51:34,133::INFO::settings.py::Setting database.user to thomas
2024-09-23 18:51:34,134::INFO::settings.py::Setting database.password to thomas


Connecting thomas@128.178.51.167:3309


2024-09-23 18:51:34,363::INFO::connection.py::Connected thomas@128.178.51.167:3309
2024-09-23 18:51:34,427::INFO::table.py::could not log event in table ~log
2024-09-23 18:51:34,713::INFO::table.py::could not log event in table ~log


In [4]:
from vr4mice.schema import vr4mice, dlc, base_analysis
import vr4mice.analysis.plotting as plotting
import vr4mice.analysis.utils as utils
import vr4mice.analysis.visual_discrim_functions as vdf
from vr4mice.analysis import regression
vdf.get_rc_params()

In [5]:
def get_trial_frames(mouse_name, date, start_time_pad = 0.5, end_time_pad = 1, start_trial=0, max_trial = 20, cam_type = "back"):
    iti = np.array((vr4mice.MouseState() & {"dataset": f"{mouse_name}_{date}_1"}).fetch("iti")[0])
    df = pd.DataFrame((vr4mice.State() & {"dataset": f"{mouse_name}_{date}_1"}).fetch("start_time", "step", "step_time", "episode", as_dict=True)[0])
    df ["iti"] = iti
    df = df[df["iti"] == 0.0].reset_index()
    result = df.groupby('episode').agg(first_step_time=('step_time', 'first'), last_step_time=('step_time', 'last')).reset_index()
    result ["first_step_time"] = result.first_step_time 
    result ["last_step_time"] = result.last_step_time 
    if cam_type == "back":
        time_steps = np.load(data_path + f"TIMESTAMPS_CamBack_{mouse_name}_{date}_1.npy") - df.start_time[0]
    if cam_type == "top":
        time_steps = np.load(data_path + f"Imagingsource_{mouse_name}_{date}_1_TS.npy") - df.start_time[0]
    list_of_frames = []
    for i in range(result.shape[0]):
        trial = result.iloc [i]
        if (trial.episode > start_trial) & (trial.episode < max_trial):
            episode_list = np.where((time_steps > trial.first_step_time - start_time_pad) & (time_steps < trial.last_step_time + end_time_pad ))[0]
            list_of_frames.append(episode_list)
    list_of_frames = np.hstack(list_of_frames)
    return(list_of_frames)

In [6]:
def get_trial_frames_both_cameras(mouse_name, date, start_time_pad = 0.5, end_time_pad = 1, start_trial=0, max_trial = 20):
    iti = np.array((vr4mice.MouseState() & {"dataset": f"{mouse_name}_{date}_1"}).fetch("iti")[0])
    df = pd.DataFrame((vr4mice.State() & {"dataset": f"{mouse_name}_{date}_1"}).fetch("start_time", "step", "step_time", "episode", as_dict=True)[0])
    df ["iti"] = iti
    df = df[df["iti"] == 0.0].reset_index()
    result = df.groupby('episode').agg(first_step_time=('step_time', 'first'), last_step_time=('step_time', 'last')).reset_index()
    result ["first_step_time"] = result.first_step_time 
    result ["last_step_time"] = result.last_step_time 
    if cam_type == "back":
        time_steps = np.load(data_path + f"TIMESTAMPS_CamBack_{mouse_name}_{date}_1.npy") - df.start_time[0]
    if cam_type == "top":
        time_steps = np.load(data_path + f"Imagingsource_{mouse_name}_{date}_1_TS.npy") - df.start_time[0]
    list_of_frames = []
    for i in range(result.shape[0]):
        trial = result.iloc [i]
        if (trial.episode > start_trial) & (trial.episode < max_trial):
            episode_list = np.where((time_steps > trial.first_step_time - start_time_pad) & (time_steps < trial.last_step_time + end_time_pad ))[0]
            list_of_frames.append(episode_list)
    list_of_frames = np.hstack(list_of_frames)
    return(list_of_frames)

In [7]:
def make_video(data_path, mouse_name, date,list_of_frames, cam_type="back", ):
    # Replace 'video_path' with the path to your video file
    if cam_type == "back":
        video_path = data_path + f'CamBack_{mouse_name}_{date}_1.avi'
        output_video_path = data_path + 'backcamera_episodes_only.mp4'  
    if cam_type == "top":
        video_path = data_path + f'Imagingsource_{mouse_name}_{date}_1_VIDEO.avi'
        output_video_path = data_path + 'topcamera_episodes_only.mp4' 

    # Initialize video capture object
    cap = cv2.VideoCapture(video_path)

    # Check if the video opened successfully
    if not cap.isOpened():
        print(f"Error: Could not open video file {video_path}")
        exit()

    # Get video properties (e.g., frame width, height, frames per second)
    frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS)

    # Create VideoWriter object to save the new video
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')  # 'mp4v' for .mp4
    out = cv2.VideoWriter(output_video_path, fourcc, fps, (frame_width, frame_height))

    # List of frames to save (this is your list_of_frames from your code)
    # Fill this with your frame indices

    frame_index = 0
    saved_frame_count = 0

    # Loop through the video and save the frames in list_of_frames to the new video
    while cap.isOpened():
        ret, frame = cap.read()
        
        if not ret:
            break  # Exit loop when no more frames are available

        if frame_index in list_of_frames:
            print(frame_index)
            # Write the frame to the output video
            out.write(frame)
            saved_frame_count += 1

        frame_index += 1

        # Optional: Stop processing if the maximum frame in list_of_frames is exceeded
        if frame_index > max(list_of_frames):
            break

    # Release video objects
    cap.release()
    out.release()
    cv2.destroyAllWindows()

    print(f"Saved {saved_frame_count} frames to video {output_video_path}.")


In [8]:
trial_frames = get_trial_frames(mouse_name=mouse_name, date=date, start_trial= 25,max_trial=100, cam_type="back")
make_video(mouse_name=mouse_name, date=date,data_path=data_path, list_of_frames=trial_frames, cam_type="back")

47150
47151
47152
47153
47154
47155
47156
47157
47158
47159
47160
47161
47162
47163
47164
47165
47166
47167
47168
47169
47170
47171
47172
47173
47174
47175
47176
47177
47178
47179
47180
47181
47182
47183
47184
47185
47186
47187
47188
47189
47190
47191
47192
47193
47194
47195
47196
47197
47198
47199
47200
47201
47202
47203
47204
47205
47206
47207
47208
47209
47210
47211
47212
47213
47214
47215
47216
47217
47218
47219
47220
47221
47222
47223
47224
47225
47226
47227
47228
47229
47230
47231
47232
47233
47234
47235
47236
47237
47238
47239
47240
47241
47242
47243
47244
47245
47246
47247
47248
47249
47250
47251
47252
47253
47254
47255
47256
47257
47258
47259
47260
47261
47262
47263
47264
47265
47266
47267
47268
47269
47270
47271
47272
47273
47274
47275
47276
47277
47278
47279
47280
47281
47282
47283
47284
47285
47286
47287
47288
47289
47290
47291
47292
47293
47294
47295
47296
47297
47298
47299
47300
47301
47302
47303
47304
47305
47306
47307
47308
47309
47310
47311
47312
47313
47314
47315
4731

In [9]:
trial_frames = get_trial_frames(mouse_name=mouse_name, date=date,start_trial= 25,max_trial=60, cam_type="back")
make_video(mouse_name=mouse_name, date=date,data_path=data_path, list_of_frames=trial_frames, cam_type="back")

47150
47151
47152
47153
47154
47155
47156
47157
47158
47159
47160
47161
47162
47163
47164
47165
47166
47167
47168
47169
47170
47171
47172
47173
47174
47175
47176
47177
47178
47179
47180
47181
47182
47183
47184
47185
47186
47187
47188
47189
47190
47191
47192
47193
47194
47195
47196
47197
47198
47199
47200
47201
47202
47203
47204
47205
47206
47207
47208
47209
47210
47211
47212
47213
47214
47215
47216
47217
47218
47219
47220
47221
47222
47223
47224
47225
47226
47227
47228
47229
47230
47231
47232
47233
47234
47235
47236
47237
47238
47239
47240
47241
47242
47243
47244
47245
47246
47247
47248
47249
47250
47251
47252
47253
47254
47255
47256
47257
47258
47259
47260
47261
47262
47263
47264
47265
47266
47267
47268
47269
47270
47271
47272
47273
47274
47275
47276
47277
47278
47279
47280
47281
47282
47283
47284
47285
47286
47287
47288
47289
47290
47291
47292
47293
47294
47295
47296
47297
47298
47299
47300
47301
47302
47303
47304
47305
47306
47307
47308
47309
47310
47311
47312
47313
47314
47315
4731

In [5]:
def get_combined_frame_both_cameras(mouse_name, date, data_path, start_time_pad = 0.5, end_time_pad = 1, start_trial=0, max_trial = 20):
    iti = np.array((vr4mice.MouseState() & {"dataset": f"{mouse_name}_{date}_1"}).fetch("iti")[0])
    df = pd.DataFrame((vr4mice.State() & {"dataset": f"{mouse_name}_{date}_1"}).fetch("start_time", "step", "step_time", "episode", as_dict=True)[0])
    df ["iti"] = iti
    df = df[df["iti"] == 0.0].reset_index()
    result = df.groupby('episode').agg(first_step_time=('step_time', 'first'), last_step_time=('step_time', 'last')).reset_index()
    result ["first_step_time"] = result.first_step_time 
    result ["last_step_time"] = result.last_step_time 


    time_steps_back = np.load(data_path + f"TIMESTAMPS_CamBack_{mouse_name}_{date}_1.npy") - df.start_time[0]
    time_steps_top = np.load(data_path + f"Imagingsource_{mouse_name}_{date}_1_TS.npy") - df.start_time[0]

    # Create an array to store the matching indices from the "top" camera
    back_camera_idx = []
    top_camera_idx = []

    last_timepoint = time_steps_top [-1]
    # For each timestamp in the back camera, find the closest timestamp in the top camera
    for i, t_back in enumerate(time_steps_back):
        if (t_back > 0) & (t_back < last_timepoint):
        # Find the index of the closest timestamp in the top camera
            closest_idx = np.abs(time_steps_top - t_back).argmin()
            back_camera_idx.append(i)
            top_camera_idx.append(closest_idx)
            
    back_camera_idx = np.array(back_camera_idx)
    top_camera_idx = np.array(top_camera_idx)

    list_of_frames_back = []
    list_of_frames_top = []
    time_steps_red_back = time_steps_back [back_camera_idx]
    for i in range(result.shape[0]):
        trial = result.iloc [i]
        if (trial.episode > start_trial) & (trial.episode < max_trial):
            episode_bool = (time_steps_red_back > trial.first_step_time - start_time_pad) & (time_steps_red_back < trial.last_step_time + end_time_pad)
            back_camera_episode_list = back_camera_idx [episode_bool]
            top_camera_episode_list = top_camera_idx [episode_bool]
            print(np.mean(time_steps_back[back_camera_episode_list] - time_steps_top[top_camera_episode_list]))
            list_of_frames_back.append(back_camera_episode_list)
            list_of_frames_top.append(top_camera_episode_list)
    list_of_frames_back = np.hstack(list_of_frames_back)
    list_of_frames_top = np.hstack(list_of_frames_top)
    return(list_of_frames_back, list_of_frames_top)

In [6]:
frames_back, frames_top = get_combined_frame_both_cameras(mouse_name=mouse_name, data_path=data_path, date=date)

-0.0030587269709660457
-0.002163248440242549
-0.0023764689763387044
-0.0030487087683949044
-0.0030187550087400527
-0.0030521555534471266
-0.0026060466108650997
-0.002056302026260731
-0.002609341713175115
-0.0020448347129444085
-0.003057345538072183
-0.0030514317889546237
-0.0030392346710994326
-0.0030224371340966992
-0.0029831761899201765
-0.002022299831269553
-0.0020623764145040065
-0.002050835545323476
-0.003004701456166632


In [53]:
time_steps_back [frames_back]

array([3.99589539e-03, 1.39963627e-02, 2.39970684e-02, ...,
       1.53464632e+02, 1.53474630e+02, 1.53484631e+02])

In [15]:
import cv2
import numpy as np

def make_side_by_side_video(data_path, mouse_name, date, list_of_frames_back, list_of_frames_top):
    """
    Combines video frames from both back and top cameras based on two lists of frame indices
    and saves them side by side.

    Parameters:
    - data_path: Path to the video files.
    - mouse_name: The name of the mouse (used to generate filenames).
    - date: Date of the video recording.
    - list_of_frames_back: List of frame indices to extract from the back camera video.
    - list_of_frames_top: List of frame indices to extract from the top camera video.
    """

    # Paths for back and top camera videos
    video_back_path = data_path + f'CamBack_{mouse_name}_{date}_1.avi'
    video_top_path = data_path + f'Imagingsource_{mouse_name}_{date}_1_VIDEO.avi'
    
    # Output path for the combined side-by-side video
    output_video_path = data_path + 'combined_video_back_top.mp4'

    # Initialize video capture objects for both videos
    cap_back = cv2.VideoCapture(video_back_path)
    cap_top = cv2.VideoCapture(video_top_path)

    # Check if the videos opened successfully
    if not cap_back.isOpened():
        print(f"Error: Could not open back camera video file {video_back_path}")
        return
    if not cap_top.isOpened():
        print(f"Error: Could not open top camera video file {video_top_path}")
        return

    # Get properties of the back camera video
    frame_width_back = int(cap_back.get(cv2.CAP_PROP_FRAME_WIDTH))
    frame_height_back = int(cap_back.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps_back = cap_back.get(cv2.CAP_PROP_FPS)

    # Get properties of the top camera video
    frame_width_top = int(cap_top.get(cv2.CAP_PROP_FRAME_WIDTH))
    frame_height_top = int(cap_top.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps_top = cap_top.get(cv2.CAP_PROP_FPS)

    # We assume the output FPS will match the lower of the two
    fps = min(fps_back, fps_top)

    # Determine the size of the output video (side-by-side combination)
    output_frame_width = frame_width_back + frame_width_top  # Combine widths
    output_frame_height = max(frame_height_back, frame_height_top)  # Use the taller height

    # Create VideoWriter object to save the new video
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')  # 'mp4v' for .mp4
    out = cv2.VideoWriter(output_video_path, fourcc, fps, (output_frame_width, output_frame_height))

    # Iterate through both lists of frames
    for back_frame_idx, top_frame_idx in zip(list_of_frames_back, list_of_frames_top):

        # Set video position to the correct frame for the back camera
        cap_back.set(cv2.CAP_PROP_POS_FRAMES, back_frame_idx)
        ret_back, frame_back = cap_back.read()

        # Set video position to the correct frame for the top camera
        cap_top.set(cv2.CAP_PROP_POS_FRAMES, top_frame_idx)
        ret_top, frame_top = cap_top.read()

        # If either frame failed to read, skip this iteration
        if not ret_back or not ret_top:
            print(f"Warning: Failed to read frames at back index {back_frame_idx} or top index {top_frame_idx}")
            continue
        
        frame_top = cv2.flip(frame_top, 0)
        frame_top = cv2.flip(frame_top, 1)

        # Resize the top frame if necessary to match the height of the back frame
        if frame_height_back != frame_height_top:
            frame_top = cv2.resize(frame_top, (frame_width_top, frame_height_back))

        # Combine the two frames side by side
        combined_frame = np.hstack((frame_back, frame_top))

        # Write the combined frame to the output video
        out.write(combined_frame)

    # Release video objects
    cap_back.release()
    cap_top.release()
    out.release()
    cv2.destroyAllWindows()

    print(f"Video saved to {output_video_path}.")



In [16]:
make_side_by_side_video(mouse_name=mouse_name, date=date,data_path=data_path, list_of_frames_back=frames_back, list_of_frames_top=frames_top)

Video saved to /Volumes/Elements/AR_sync_videos/pheasent_1408_dicrim_no_occluder/combined_video_back_top.mp4.


array([3.99589539e-03, 1.39963627e-02, 2.39970684e-02, ...,
       1.53464632e+02, 1.53474630e+02, 1.53484631e+02])

In [60]:
time_steps_top [frames_top]

array([1.99699402e-03, 1.19979382e-02, 2.19955444e-02, ...,
       1.53461631e+02, 1.53472631e+02, 1.53481627e+02])

In [62]:
len(frames_back)

4323